# Results: Calibration & OOD Variance

Run after:
1. `python scripts/train.py --out checkpoints/bert_sst2`
2. `python scripts/eval_mc_dropout.py --checkpoint checkpoints/bert_sst2 --out results/mc_predictions.npz`
3. `python scripts/eval_calibration.py --predictions results/mc_predictions.npz --out results/calibration.npz`

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if (Path.cwd().parent / "src").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
RESULTS_DIR = ROOT / "results"

## 1. Reliability diagram (calibration)

In [ ]:
cal = np.load(RESULTS_DIR / "calibration.npz", allow_pickle=True)
bin_accs, bin_confs, bin_counts, bins = cal["bin_accs"], cal["bin_confs"], cal["bin_counts"], cal["bins"]
ece = float(cal["ece"])

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.bar(bins[:-1], bin_accs, width=1/len(bins), align="edge", alpha=0.7, label="Accuracy")
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Confidence (mean P(positive))")
ax.set_ylabel("Accuracy")
ax.set_title(f"Reliability diagram (ECE = {ece:.3f})")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 2. OOD variance comparison (ID vs Wikitext vs Corrupted)

In [ ]:
mc = np.load(RESULTS_DIR / "mc_predictions.npz", allow_pickle=True)
id_var = mc["id_var_positive"]
ood_var = mc["ood_wikitext_var"]
corr_var = mc["corr_var_positive"]

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.boxplot([id_var, ood_var, corr_var], labels=["ID (SST-2)", "OOD (Wikitext)", "Corrupted SST-2"])
ax.set_ylabel("Predictive variance σ²")
ax.set_title("MC Dropout variance: ID vs OOD")
plt.tight_layout()
plt.show()

print(f"ID    mean(var) = {id_var.mean():.4f}")
print(f"OOD   mean(var) = {ood_var.mean():.4f}  ratio = {ood_var.mean()/id_var.mean():.2f}x")
print(f"Corr  mean(var) = {corr_var.mean():.4f}  ratio = {corr_var.mean()/id_var.mean():.2f}x")